# Практика · GloVe і FastText

> Лекція: [lecture.html](lecture.html) · Домашнє завдання: [homework.html](homework.html) ·
> Тест: [quiz.html](quiz.html)

Тут ми рахуємо **всі** числа, які називає лекція, і в тому самому порядку.

Що зробимо:

1. Зберемо той самий корпус українських перекладів, що й у попередніх темах, і
   подивимось, скільки словоформ модель **не бачить** при порозі `min_count = 10`.
2. Побудуємо матрицю співвіднесень і напишемо **GloVe з нуля** — разом із ваговою
   функцією, без якої він не працює.
3. Заміряємо, чи видно різницю між GloVe і Word2Vec **на нашому корпусі**, і
   окремо — скільки коштує вибір, яку з двох матриць вважати вектором слова.
4. Напишемо **субслова** (n-грами) і зберемо з них вектор слова, якого модель не бачила.
5. Порівняємо дві оцінки користі субслів: **оптимістичну** (перекриття n-грам) і
   **чесну** (чи справді відома лема) — вони розходяться майже втричі.
6. Заміряємо користь субслів **на самій задачі**, а не на перекритті.
7. Знайдемо в корпусі місця, де субслова **шкодять**.

> ⏱ Зошит навчає девʼять моделей GloVe, три skip-gram і три FastText.
> Заміряно на одному ядрі без відеокарти: близько **трьох хвилин** процесорного
> часу — пʼять прогонів дали від **156** до **204 с**. За стінним годинником ті
> самі прогони тривали від **2 хв 54 с** до **8 хв**, залежно від того, скільки
> зошитів рахувалося на машині поруч. Саме тому час ми міряємо
> `time.process_time()`, а не годинником на стіні.

## 1 · Середовище й запобіжник на потоки

Перша клітинка друкує версії — якщо в тебе інші, числа можуть трохи поїхати.

І одразу найважливіший рядок усього блоку: **фіксація потоків до імпорту numpy**.
Без неї бібліотеки лінійної алгебри запускають потоки OpenMP, ті крутяться в
очікуванні, і це очікування рахується як робота. Процесорний час у такому режимі
завищений у десятки разів, тобто всі наші заміри «скільки коштує навчання» стали б
вигадкою.

In [ ]:
import os

# ⚠️ ці чотири рядки мусять стояти ДО імпорту numpy, інакше час навчання бреше
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'

import sys, re, math, glob, gettext, time, collections, random
import numpy as np
import scipy.sparse as sparse
import sklearn
import pymorphy3

print("Python    ", sys.version.split()[0])
print("numpy     ", np.__version__)
print("scipy     ", __import__("scipy").__version__)
print("sklearn   ", sklearn.__version__)
print("pymorphy3 ", pymorphy3.__version__)
print()
NOTEBOOK_START = time.process_time()
print("потоки зафіксовано:", os.environ['OMP_NUM_THREADS'], "— час міряємо time.process_time()")

## 2 · Корпус: ті самі українські переклади інтерфейсів

Корпус спільний для всього курсу: пари «англійський оригінал → український
переклад» із файлів локалізації, які вже лежать на диску. Жодної мережі, жодної
випадковості між запусками.

Якщо української локалі на машині немає, вмикається маленький вбудований корпус —
зошит виконається до кінця, але числа будуть інші, і він про це скаже.

In [ ]:
def load_system_corpus():
    """Читаємо всі .mo-файли української локалі.
    Повертаємо трійки (програма, англійський оригінал, український переклад)."""
    docs = []
    for path in sorted(glob.glob('/usr/share/locale/uk/LC_MESSAGES/*.mo')):
        try:
            with open(path, 'rb') as f:
                catalog = gettext.GNUTranslations(f)
        except Exception:
            continue                      # зламаний або чужий формат — просто пропускаємо
        program = path.split('/')[-1][:-3]
        for source, target in catalog._catalog.items():
            # службовий заголовок каталогу має ключ '' і не є текстом
            if isinstance(source, str) and isinstance(target, str) \
               and len(target) > 30 and 'Project-Id' not in target:
                docs.append((program, source, target))
    return docs


FALLBACK = [
"перекодувати метадані в це кодування",
"Перевищено час очікування на введення-виведення з гнізда",
"Буде проігноровано, зберігається для сумісності",
"Не вдалося отримати дані про ємність акумулятора",
"Додає маломасштабний растр, подібний до локальної зернистості",
"Налаштувати таблицю клавіш кани",
"Додати тло документа до кожного перетвореного шару.",
"Обмежити загальне використання чорнил на аркуш паперу",
"Вікно віджету, яке реалізовано",
"Мінімальне розширення дочірнього елемента",
"Вам слід включити принаймні один диск як ціль встановлення.",
"швейцарська німецька мова жестів",
"Виконано забагато спроб отримання пароля",
"Андаманські і Нікобарські острови",
"Чи слід підвкладкам заповнювати виділену ділянку",
"Увімкнути помилку подвійного подавання через товщину паперу",
"Однорідний за розміром вертикально",
"Некоректний формат прав доступу",
"Неочікуване завершення формального виразу",
"Більше старих транзакцій немає.",
"Некоректні параметри режиму знімання",
"Тритлумити яскравість екрана після періоду неактивності",
"Використовувати спекуляцію даних після перезавантаження.",
"вказана адреса основного сервера ключів є некоректною",
"не вдалося створити знімок вікна",
"Порогове значення порожніх сторінок для програмного відкидання",
"не перевіряти шляхи символічних посилань на файли",
"Збільшений час очікування лампи",
"не є коректною числовою специфікацією",
"пароль користувача у форматі звичайного тексту",
"не вдалося отримати повідомлення від батьківського процесу",
"Точне налаштування балансу білого",
"Ширина паперу, на якому слід надрукувати дані",
"показувати також ідентифікатори груп",
"УВАГА: цілісність повідомлення не захищено",
"Опале листя восени або щось вкрите живим листям",
"Під час приготування документа сталася помилка",
"помилка у додатку підтвердження",
"Відстань за вертикаллю між двома дочірніми віджетами",
"не можна комбінувати кілька визначників фільтра",
"Не вдалося оновити файл налаштування.",
"в заголовку файлу не знайдено магічного рядка",
"Динамічна область, найближчий предмет",
"несумісні типи в бінарному виразі",
"Не вдалося обробити відповідь сервера",
"Не вдалося побудувати назву інтерфейсу",
"Визначає, чи може бути встановлено часткову відповідність",
"вивести список усіх пакунків неактивних потоків модуля",
"не вказано модель захисту під час використання декількох міток",
"Показати список відомих областей",
"Не вдалося отримати список мовних розширень модуля",
"Не вдалося завантажити модуль обробки",
"Назва групи для перетягування вкладки",
"Неочікуваний передчасний кінець потоку",
"Датчик кольорової ділянки з двома чипами",
"Вивести відомості щодо користувача і перевірити розпізнавання",
"Ідентифікатор повідомлення у каталозі",
"Перемістити обчислення незмінних у петлях за межі петель.",
"Не вдалося виконати читання з монітора",
"Мінімальна ширина кнопок в контейнері",
"параметри містив пароль з порожньою назвою",
"Буде встановлено при наступному вході",
"Попереджати про невіртуальні деструктори.",
"Показати коротке повідомлення щодо користування для команди",
"Апаратний блок призначення не збігається із джерелом",
"Ізольована мережа, лише внутрішня маршрутизація",
"Висота відеозапису, одержаного з камери, в пікселях",
"Процесором основної системи не надаються потрібні можливості",
"Немає повідомлення бажаного типу",
"Чи інтеграцію з оболонкою ввімкнено",
"Показувати під час запуску стовпчик використання процесору.",
"Не можна вилучати загальносистемні розширення",
"Команда автоматичної компенсації спалаху",
"Чи показується розмір вибраного шрифту у позначці",
"Не вдалося створити зображення попереднього перегляду друку.",
"кореневий сертифікат було позначено як надійний",
"Перейти до попереднього елемента списку",
"Чи показувати кнопку закривання на панелі інструментів",
"Базовий рельєф з двома джерелами світла",
"недійсні типи у конвертації до цілого числа",
"Не вдалося виконати пошук файла",
"У назві домену не повинно міститися символів розриву рядка",
"регістр джерела збігається з основою зворотного запису",
"не вдалося завантажити дані символів",
"Експортувати обрізаний вміст у сторінці",
"читання користувацьких методів доступу",
"Масштабувати за центром сторінки",
"перевизначити типове розташування кореневої теки системи",
"обʼєкт призначення виділяється тут",
"Визначає відстань до точки призначення.",
"Під час спроби вивантаження підшляху метаданих:",
"Для встановлення усіх наданих випусків потрібен пристрій",
"вивести дані у зручному для читання форматі",
"Не вдалося отримати адреси фрагмента.",
"елемент масиву позначок не є рядком",
"Внутрішня помилка: невідома помилка",
"Автономний регіон Азорські острови",
"Встановити базовою одиницею виміру сантиметр",
"Рядок, що показано в позначці вкладки дочірнього елемента",
"Визначає спосіб малювання тіні навколо порту перегляду",
"помилка позиціонування голівки сканування",
"встановити ширину каналу перенесення після копіювання",
"Керування динамічними правами доступу",
"Друкувати виконані операції і код завершення команди",
"Неможливо започаткувати аналізатор файлів конфігурації.",
"Показати додаткові діагностичні дані",
"Прочитати кутку зі стандартного джерела вхідних даних",
"визначити батьківський запис поточного знімка",
"Не можна мати розділи, які перекриваються.",
"ігноруємо некоректний необроблений пароль",
"Неможливо перейменувати файли верхнього рівня",
"конфліктуючі рядки заміни для порожнього поля",
"Глянцевий фотопапір найвищої якості",
"мало бути вказано регістр цілих чисел",
"Клавіатурне скорочення для очищення підсвічування знайденого",
"Використовувати лише апаратні значення",
"В цьому файлі немає динамічного розділу.",
"Виберіть вузол для отримання форми входу",
"відповідного пристрою файлової системи не знайдено",
"Файл сертифіката для перевірки сервера",
]

corpus = load_system_corpus()
SYSTEM_CORPUS = len(corpus) >= 5000
if not SYSTEM_CORPUS:
    corpus = [('fallback', '', text) for text in FALLBACK]
    print("⚠️  системної локалі немає — працюємо на вбудованому корпусі, числа будуть інші")
else:
    print("шлях спрацював: /usr/share/locale/uk/LC_MESSAGES/*.mo")

print("документів:", len(corpus))
print("програм:   ", len(set(program for program, _, _ in corpus)))
print()
for program, source, target in corpus[:2]:
    print(f"[{program}] {source[:44]!r}\n         -> {target[:58]!r}")

## 3 · Токенізатор — канон курсу, не свій

Шаблон той самий, що в темах 01-12. Апостроф у ньому — **звʼязка всередині слова**,
а не окремий символ, тож «зʼєднання» лишається одним токеном.

Брати свій шаблон не можна: різниця в пів відсотка словника розсипає числа, які
теми цитують одна в одної.

In [ ]:
TOKEN_PATTERN = r"[а-яїієґ]+(?:['ʼ’][а-яїієґ]+)*"
tokenize = re.compile(TOKEN_PATTERN).findall

documents = [target for _, _, target in corpus]
sentences = [tokenize(text.lower()) for text in documents]

word_counts = collections.Counter()
for sentence in sentences:
    word_counts.update(sentence)

total_tokens = sum(len(sentence) for sentence in sentences)
print("слововживань:", total_tokens)
print("словоформ:   ", len(word_counts))
print()
print("текст :", documents[17])
print("токени:", sentences[17])

## 4 · Поріг `min_count`: чого модель не побачить ніколи

Будь-яка модель ембедингів має поріг: слово, що трапилось рідше за `min_count`
разів, до словника не потрапляє. Причина проста — на пʼятьох прикладах вектор
вивчити неможливо, вийде шум.

Питання в тому, скільки мови цей поріг відрізає. І відповідь у двох числах,
які треба дивитись **разом**: скільки відрізано **словоформ** і скільки
відрізано **слововживань**.

In [ ]:
print(f"{'min_count':>10} {'словник':>9} {'покриття':>10} {'невідомих форм':>16} {'невідомих вживань':>19}")
for threshold in (5, 10, 20, 50):
    known = {word for word, count in word_counts.items() if count >= threshold}
    kept = sum(count for word, count in word_counts.items() if count >= threshold)
    unknown_types = len(word_counts) - len(known)
    print(f"{threshold:>10} {len(known):>9} {100 * kept / total_tokens:>9.2f} % "
          f"{unknown_types:>9} ({100 * unknown_types / len(word_counts):>4.1f} %) "
          f"{100 * (total_tokens - kept) / total_tokens:>15.2f} %")

MIN_COUNT = 10 if SYSTEM_CORPUS else 2
vocabulary = sorted((word for word, count in word_counts.items() if count >= MIN_COUNT),
                    key=lambda word: -word_counts[word])
word_index = {word: i for i, word in enumerate(vocabulary)}
known_words = set(vocabulary)
oov_words = sorted(word for word in word_counts if word not in known_words)

print()
print(f"беремо min_count = {MIN_COUNT}: словник {len(vocabulary)} слів, "
      f"поза словником {len(oov_words)} словоформ")

## 5 · Дві дороги до вектора: пари й матриця

Word2Vec (тема 12) ходить по **парах**: узяв слово, узяв сусіда, підправив два
вектори, пішов далі. GloVe починає з іншого кінця — він спершу складає **таблицю
співвіднесень**: скільки разів слово `i` траплялось поруч зі словом `j` у всьому
корпусі. Далі корпус більше не потрібен: GloVe працює вже з таблицею.

Порахуємо обидві величини на нашому корпусі: скільки виходить пар і скільки
непорожніх клітинок у таблиці. Вікно — пʼять слів ліворуч і пʼять праворуч.

In [ ]:
WINDOW = 5

sentence_ids = [np.array([word_index[word] for word in sentence if word in word_index],
                         dtype=np.int32)
                for sentence in sentences]
sentence_ids = [ids for ids in sentence_ids if len(ids) > 1]

def build_pairs(window=WINDOW):
    """Усі пари (центральне слово, сусід) — саме те, чим годують skip-gram."""
    centers, contexts = [], []
    for ids in sentence_ids:
        n = len(ids)
        for i in range(n):
            for j in range(max(0, i - window), min(n, i + window + 1)):
                if j != i:
                    centers.append(ids[i])
                    contexts.append(ids[j])
    return np.array(centers, dtype=np.int32), np.array(contexts, dtype=np.int32)

started = time.process_time()
pair_center, pair_context = build_pairs()
print(f"документів у роботі: {len(sentence_ids)}")
print(f"пар (слово, сусід):  {len(pair_center)}")
print(f"зібрано за {time.process_time() - started:.1f} с процесорного часу")

Тепер та сама інформація у вигляді таблиці. Одна деталь: далекий сусід важить
менше за близького, тому кожна пара додає в клітинку не одиницю, а `1/d`, де
`d` — відстань у словах. Це рецепт самого GloVe, і він же пояснює, чому в
таблиці зʼявляються дробові числа.

In [ ]:
started = time.process_time()
cooccurrence = collections.defaultdict(float)
for ids in sentence_ids:
    n = len(ids)
    for i in range(n):
        for j in range(max(0, i - WINDOW), i):
            weight = 1.0 / (i - j)          # далекий сусід важить менше за близького
            cooccurrence[(ids[i], ids[j])] += weight
            cooccurrence[(ids[j], ids[i])] += weight

cooc_keys = np.array(list(cooccurrence.keys()), dtype=np.int32)
cooc_left = cooc_keys[:, 0].copy()
cooc_right = cooc_keys[:, 1].copy()
cooc_value = np.array(list(cooccurrence.values()), dtype=np.float32)

cells_total = len(vocabulary) ** 2
print(f"розмір таблиці:      {len(vocabulary)} x {len(vocabulary)} = {cells_total} клітинок")
print(f"непорожніх клітинок: {len(cooc_value)} ({100 * len(cooc_value) / cells_total:.2f} %)")
print(f"уся вага в таблиці:  {cooc_value.sum():.1f}")
print(f"найбільша клітинка:  {cooc_value.max():.1f}")
print(f"зібрано за {time.process_time() - started:.1f} с процесорного часу")
print()
print("три пари, які найчастіше стоять поруч:")
for k in np.argsort(-cooc_value)[:6:2]:
    print(f"   {vocabulary[cooc_left[k]]:<12} + {vocabulary[cooc_right[k]]:<12} X = {cooc_value[k]:8.1f}")

Порівняй два числа: **пар** утричі більше, ніж **клітинок**. Це і є перша
відмінність двох методів, і вона не про якість, а про роботу: skip-gram
проходить кожну пару окремо, GloVe складає однакові пари в одну клітинку й далі
працює тільки з нею.

Ще одне число, потрібне далі: розподіл значень у таблиці страшенно нерівний.

In [ ]:
sorted_values = np.sort(cooc_value)[::-1]
one_percent = max(1, len(sorted_values) // 100)
print(f"клітинок зі значенням X < 1:    {100 * np.mean(cooc_value < 1):.1f} %")
print(f"клітинок зі значенням X >= 10:  {100 * np.mean(cooc_value >= 10):.2f} %")
print(f"верхній 1 % клітинок тримає:    {100 * sorted_values[:one_percent].sum() / cooc_value.sum():.1f} % усієї ваги")
print()
print("гістограма значень X (для фігури лекції):")
edges = [0.0, 0.5, 1.0, 2.0, 5.0, 10.0, 50.0, 100.0, 1000.0, 100000.0]
for low, high in zip(edges[:-1], edges[1:]):
    part = np.mean((cooc_value >= low) & (cooc_value < high))
    print(f"   X у [{low:>6.1f}, {high:>8.1f}) : {100 * part:5.2f} %")

## 6 · Формула GloVe і вагова функція

GloVe вимагає, щоб скалярний добуток двох векторів дорівнював логарифму того,
скільки разів слова стояли поруч:

    w_i · w~_j + b_i + b~_j  ≈  log X_ij

Помилку кожної клітинки він підносить до квадрата і **зважує** функцією `f(X)`:

    f(x) = (x / x_max)^0.75, якщо x < x_max, інакше 1

Навіщо вага. Клітинок зі значенням «одна десята» — більшість, і кожна з них
майже нічого не значить: слова стояли поруч один раз на далекій відстані. Якби
всі клітинки важили однаково, мільйон випадковостей заглушив би кілька тисяч
справжніх звʼязків.

Навіщо **обрізання** (`min` з одиницею). Найбільша клітинка нашої таблиці
більша за десятку в сотні разів. Без обрізання її вага так само була б у десятки
разів більшою — і вся модель підлаштувалась би під кілька пар службових слів.

Порахуємо це руками.

In [ ]:
X_MAX, ALPHA = 10.0, 0.75

def weight_clipped(x):
    """Канонічна вага GloVe: росте до x_max, далі стала."""
    return np.minimum(1.0, (x / X_MAX) ** ALPHA)

def weight_unclipped(x):
    """Та сама формула, але без обрізання — для порівняння."""
    return (x / X_MAX) ** ALPHA

biggest = float(cooc_value.max())
print(f"{'X':>10} {'f(X) з обрізанням':>20} {'f(X) без обрізання':>21}")
for x in (0.2, 1.0, 2.0, 5.0, 10.0, 100.0, biggest):
    print(f"{x:>10.1f} {float(weight_clipped(x)):>20.4f} {float(weight_unclipped(x)):>21.4f}")
print()
print(f"без обрізання найчастіша пара важить у "
      f"{float(weight_unclipped(biggest)) / float(weight_unclipped(10.0)):.0f} разів більше за пару з X = 10")
print(f"з обрізанням — рівно в {float(weight_clipped(biggest)) / float(weight_clipped(10.0)):.0f} раз")

## 7 · Пишемо GloVe

Бібліотеки `gensim` тут немає, і це на краще: формула має бути видима.

Навчання йде методом AdaGrad — це той самий градієнтний спуск, але з памʼяттю:
кожна координата має власний крок, який тим менший, чим більше ця координата вже
рухалась. Для GloVe це важливо, бо частота слів різниться в тисячі разів.

In [ ]:
def scatter_add(target, rows, values):
    """target[rows[k]] += values[k], з правильним складанням повторів.
    Через розріджену матрицю це виходить у кілька разів швидше за np.add.at."""
    m = len(rows)
    picker = sparse.csr_matrix((np.ones(m, dtype=np.float32), (rows, np.arange(m))),
                               shape=(target.shape[0], m))
    target += picker @ values


def train_glove(seed, dim=64, epochs=6, batch=32768, lr=0.15, clipped=True, weighted=True):
    """GloVe з нуля. clipped=False — вага без обрізання, weighted=False — без ваги взагалі."""
    rng = np.random.default_rng(seed)
    size = len(vocabulary)
    main = ((rng.random((size, dim), dtype=np.float32) - 0.5) / dim).astype(np.float32)
    ctx = ((rng.random((size, dim), dtype=np.float32) - 0.5) / dim).astype(np.float32)
    bias_main = np.zeros(size, dtype=np.float32)
    bias_ctx = np.zeros(size, dtype=np.float32)
    # накопичені квадрати градієнтів — памʼять AdaGrad
    acc_main = np.ones_like(main)
    acc_ctx = np.ones_like(ctx)
    acc_bias_main = np.ones(size, dtype=np.float32)
    acc_bias_ctx = np.ones(size, dtype=np.float32)

    if not weighted:
        weights = np.ones_like(cooc_value)
    elif clipped:
        weights = weight_clipped(cooc_value).astype(np.float32)
    else:
        weights = weight_unclipped(cooc_value).astype(np.float32)
    log_x = np.log(cooc_value)

    n = len(cooc_value)
    for _ in range(epochs):
        order = rng.permutation(n)
        for start in range(0, n, batch):
            k = order[start:start + batch]
            i, j = cooc_left[k], cooc_right[k]
            vi, vj = main[i], ctx[j]
            # наскільки скалярний добуток розходиться з логарифмом частоти
            error = (vi * vj).sum(1) + bias_main[i] + bias_ctx[j] - log_x[k]
            scaled = weights[k] * error
            grad_i = (scaled[:, None] * vj).astype(np.float32)
            grad_j = (scaled[:, None] * vi).astype(np.float32)
            scatter_add(acc_main, i, grad_i * grad_i)
            scatter_add(acc_ctx, j, grad_j * grad_j)
            scatter_add(main, i, (-lr * grad_i / np.sqrt(acc_main[i])).astype(np.float32))
            scatter_add(ctx, j, (-lr * grad_j / np.sqrt(acc_ctx[j])).astype(np.float32))
            grad_bias = scaled.astype(np.float32)
            np.add.at(acc_bias_main, i, grad_bias * grad_bias)
            np.add.at(acc_bias_ctx, j, grad_bias * grad_bias)
            np.add.at(bias_main, i, -lr * grad_bias / np.sqrt(acc_bias_main[i]))
            np.add.at(bias_ctx, j, -lr * grad_bias / np.sqrt(acc_bias_ctx[j]))
    # повертаємо все, що знадобиться далі: два набори векторів і два зсуви
    return main, ctx, bias_main, bias_ctx


# Який бік розкладу брати — окреме рішення, і воно мусить бути однакове для всіх
# моделей теми. Інакше ми порівнюватимемо не методи, а деталь реалізації.
# Розділ 12 показує, скільки коштує цей вибір; тут фіксуємо центральну матрицю.
def glove_vectors(model, side='main'):
    """side='main' — центральна матриця, 'ctx' — контекстна, 'sum' — сума двох."""
    if side == 'main':
        return model[0]
    if side == 'ctx':
        return model[1]
    return model[0] + model[1]

print("GloVe написано:", train_glove.__doc__.splitlines()[0])

Перш ніж вірити навчанню, перевіримо саму похідну. Візьмемо одну клітинку,
порахуємо градієнт формулою — і чисельно, зсунувши координату на мільйонну
частку. Якщо реалізація правильна, обидва числа збігатимуться до шостого знака.

In [ ]:
rng = np.random.default_rng(0)
test_i = rng.random(4).astype(np.float64)
test_j = rng.random(4).astype(np.float64)
test_x, test_bias = 7.0, 0.3

def glove_loss(vec_i, vec_j):
    """Внесок однієї клітинки в цільову функцію GloVe."""
    error = float(np.dot(vec_i, vec_j)) + test_bias + test_bias - math.log(test_x)
    return float(weight_clipped(test_x)) * error ** 2

analytic = 2 * float(weight_clipped(test_x)) * (
    float(np.dot(test_i, test_j)) + 2 * test_bias - math.log(test_x)) * test_j
step = 1e-6
numeric = np.zeros(4)
for k in range(4):
    moved = test_i.copy()
    moved[k] += step
    numeric[k] = (glove_loss(moved, test_j) - glove_loss(test_i, test_j)) / step

print("градієнт формулою :", np.round(analytic, 6))
print("градієнт чисельно :", np.round(numeric, 6))
assert np.allclose(analytic, numeric, atol=1e-5), "похідна GloVe розійшлася!"
print("✅ похідна збігається")

## 8 · Задача, на якій міряємо: знайти той самий зміст іншими словами

Порівнювати моделі за списками сусідів — заняття приємне, але недоказове. Треба
задача з числом.

Нам знову щастить із корпусом: той самий англійський рядок різні перекладачі
переклали по-різному, тож ми маємо пари документів, про які **точно** відомо, що
вони означають одне й те саме. Тема 05 уже брала ці пари й показала, що TF-IDF
їх майже не бачить.

Задача така: беремо перший документ пари, кладемо другий у купу з пʼятисот
випадкових документів корпусу і дивимось, на якому місці модель його поставить.
Метрика — **MRR**: одиниця поділена на місце правильної відповіді, усереднена по
всіх парах. Одиниця означає «завжди перший», нуль — «не знаходить ніколи».

In [ ]:
NOISE = re.compile(r"[@<>]|https?://")

def paraphrase_pairs():
    """Пари українських перекладів того самого англійського рядка,
    у яких майже немає спільних слів."""
    by_source = collections.defaultdict(dict)
    for program, source, target in corpus:
        if source.strip() == 'translator-credits':
            continue
        by_source[source][' '.join(target.split())] = program
    found = []
    for source, variants in by_source.items():
        texts = list(variants)
        for i in range(len(texts)):
            for j in range(i + 1, len(texts)):
                first, second = texts[i], texts[j]
                if NOISE.search(first) or NOISE.search(second):
                    continue
                a = set(tokenize(first.lower()))
                b = set(tokenize(second.lower()))
                if len(a) < 3 or len(b) < 3:
                    continue
                if len(a & b) / len(a | b) < 0.34:
                    found.append((first, second))
    return found


HANDMADE_PAIRS = [
    ("Не вдалося зберегти файл на диск", "Запис файла на носій завершився невдало"),
    ("Пароль введено неправильно", "Хибний пароль користувача"),
    ("Перевірте зʼєднання з мережею", "Немає доступу до інтернету"),
    ("Оберіть теку для збереження", "Вкажіть каталог призначення"),
    ("Програма несподівано завершила роботу", "Застосунок аварійно закрився"),
    ("Оновлення встановлено успішно", "Нову версію застосовано без помилок"),
]

pairs = paraphrase_pairs()
if len(pairs) < 20:
    pairs = HANDMADE_PAIRS
    print("⚠️  у корпусі бракує подвійних перекладів — беремо вбудований список пар")
print("пар «те саме іншими словами»:", len(pairs))
oov_in_pairs = sum(1 for a, b in pairs
                   if any(w not in known_words for w in tokenize(a.lower()) + tokenize(b.lower())))
tokens_in_pairs = sum(len(tokenize(a.lower())) + len(tokenize(b.lower())) for a, b in pairs)
oov_tokens = sum(1 for a, b in pairs
                 for w in tokenize(a.lower()) + tokenize(b.lower()) if w not in known_words)
print(f"пар, де є хоч одне слово поза словником: {oov_in_pairs}")
print(f"слововживань у парах: {tokens_in_pairs}, з них поза словником {oov_tokens} "
      f"({100 * oov_tokens / tokens_in_pairs:.1f} %)")
print()
for a, b in pairs[:2]:
    print("   A:", a)
    print("   B:", b)

In [ ]:
def document_vector(text_tokens, vectors, unknown_vector=None):
    """Вектор документа — середнє векторів його слів.
    unknown_vector: як діставати вектор слова поза словником (None — пропускати)."""
    picked = []
    for word in text_tokens:
        if word in word_index:
            picked.append(vectors[word_index[word]])
        elif unknown_vector is not None:
            guess = unknown_vector(word)
            if guess is not None:
                picked.append(guess)
    if not picked:
        return None
    middle = np.mean(picked, axis=0)
    length = np.linalg.norm(middle)
    return middle / length if length > 0 else None


DISTRACTORS = 500
distractor_rng = np.random.default_rng(0)
long_enough = [words for words in sentences if len(words) >= 3]
distractor_ids = distractor_rng.choice(len(long_enough), min(DISTRACTORS, len(long_enough)),
                                       replace=False)
distractor_texts = [long_enough[k] for k in distractor_ids]


def retrieval_score(vectors, unknown_vector=None, subset=None):
    """MRR: на якому місці модель ставить другий документ пари серед 500 чужих."""
    chosen = pairs if subset is None else [pairs[i] for i in subset]
    queries, answers = [], []
    for a, b in chosen:
        va = document_vector(tokenize(a.lower()), vectors, unknown_vector)
        vb = document_vector(tokenize(b.lower()), vectors, unknown_vector)
        if va is not None and vb is not None:
            queries.append(va)
            answers.append(vb)
    noise = [document_vector(text, vectors, unknown_vector) for text in distractor_texts]
    noise = np.array([v for v in noise if v is not None], dtype=np.float32)
    queries = np.array(queries, dtype=np.float32)
    answers = np.array(answers, dtype=np.float32)
    right_score = (queries * answers).sum(1)
    noise_score = queries @ noise.T
    ranks = 1 + (noise_score > right_score[:, None]).sum(1)
    return float(np.mean(1.0 / ranks)), float(np.mean(ranks == 1)), len(ranks)


def nearest(vectors, word, how_many=5):
    """Найближчі за косинусом слова словника."""
    unit = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)
    i = word_index[word]
    scores = unit @ unit[i]
    scores[i] = -9
    return [vocabulary[j] for j in np.argsort(-scores)[:how_many]]

# швидка перевірка, що наш косинус — це саме косинус
from sklearn.metrics.pairwise import cosine_similarity
probe = np.random.default_rng(1).random((4, 8)).astype(np.float32)
ours = (probe / np.linalg.norm(probe, axis=1, keepdims=True)) @ \
       (probe / np.linalg.norm(probe, axis=1, keepdims=True)).T
assert np.allclose(ours, cosine_similarity(probe), atol=1e-6), "косинус розійшовся!"
print("✅ наш косинус збігається з sklearn.cosine_similarity")
print("документів-відволікачів:", len(distractor_texts))

## 9 · GloVe: три зерна

Одне зерно нічого не доводить. Навчаємо три моделі й дивимось не лише на середнє,
а й на розкид: різниця, менша за розкид, — це не різниця.

In [ ]:
SEEDS = (0, 1, 2)
glove_scores, glove_times, glove_models = [], [], []
for seed in SEEDS:
    started = time.process_time()
    model = train_glove(seed)
    spent = time.process_time() - started
    vectors = glove_vectors(model)
    mrr, top1, used = retrieval_score(vectors)
    glove_scores.append(mrr)
    glove_times.append(spent)
    glove_models.append(model)
    print(f"зерно {seed}: MRR {mrr:.4f}, перший з першого разу {100 * top1:.1f} %, "
          f"навчання {spent:.1f} с")

print()
print(f"GloVe: MRR {np.mean(glove_scores):.4f} ± {np.std(glove_scores):.4f}, "
      f"навчання {np.mean(glove_times):.1f} с")
print()
glove_first = glove_vectors(glove_models[0])
for word in ('файл', 'помилка', 'вікно', 'мережі'):
    print(f"   {word:<9} -> {' · '.join(nearest(glove_first, word))}")

## 10 · Що буває без вагової функції

Тепер найцікавіше в GloVe. Проженемо ту саму модель тричі трьома способами:

- **з обрізанням** — канонічна `f(x)`;
- **без обрізання** — та сама формула, але вага росте без стелі;
- **без ваги взагалі** — усі клітинки важать однаково.

Усе решта однакове: ті самі зерна, та сама кількість проходів.

In [ ]:
variants = {
    "f(x) з обрізанням": dict(clipped=True, weighted=True),
    "f(x) без обрізання": dict(clipped=False, weighted=True),
    "без ваги (f = 1)": dict(clipped=True, weighted=False),
}
variant_scores = {"f(x) з обрізанням": glove_scores}
variant_models = {"f(x) з обрізанням": glove_models[0]}
for name, options in variants.items():
    if name in variant_scores:
        continue
    scores = []
    for seed in SEEDS:
        model = train_glove(seed, **options)
        mrr, _, _ = retrieval_score(glove_vectors(model))
        scores.append(mrr)
        if seed == 0:
            variant_models[name] = model
            print(f"{name}: сусіди слова «файл» -> "
                  f"{' · '.join(nearest(glove_vectors(model), 'файл'))}")
    variant_scores[name] = scores

print()
for name, scores in variant_scores.items():
    print(f"{name:<20} MRR {np.mean(scores):.4f} ± {np.std(scores):.4f}")
best = max(variant_scores, key=lambda k: np.mean(variant_scores[k]))
print()
print(f"найкращий варіант: {best}")

Різниця між варіантами на задачі виявилась маленькою — і це саме той випадок, коли
треба подивитись, що вага робить **усередині**, а не лише що вона дає назовні.

Порахуємо, наскільки добре кожна модель відтворює `log X` — окремо для трьох куп
клітинок: майже порожніх, середніх і великих. Саме сюди вага й перерозподіляє
зусилля моделі.

In [ ]:
def reconstruction_error(model, sample=200000, seed=0):
    """Корінь із середньої квадратичної похибки log X, окремо по купах клітинок."""
    main, ctx, bias_main, bias_ctx = model
    rng = np.random.default_rng(seed)
    k = rng.choice(len(cooc_value), min(sample, len(cooc_value)), replace=False)
    left, right = cooc_left[k], cooc_right[k]
    predicted = (main[left] * ctx[right]).sum(1) + bias_main[left] + bias_ctx[right]
    error = predicted - np.log(cooc_value[k])
    out = []
    for low, high in ((0.0, 1.0), (1.0, 10.0), (10.0, 1e9)):
        inside = (cooc_value[k] >= low) & (cooc_value[k] < high)
        out.append(math.sqrt(float(np.mean(error[inside] ** 2))))
    return out

print(f"{'варіант':<22} {'X < 1':>9} {'1 <= X < 10':>13} {'X >= 10':>10}")
for name in ("f(x) з обрізанням", "f(x) без обрізання", "без ваги (f = 1)"):
    errors = reconstruction_error(variant_models[name])
    print(f"{name:<22} {errors[0]:>9.4f} {errors[1]:>13.4f} {errors[2]:>10.4f}")
print()
print("частка клітинок у купах: "
      f"X < 1 — {100 * np.mean(cooc_value < 1):.1f} %, "
      f"1 <= X < 10 — {100 * np.mean((cooc_value >= 1) & (cooc_value < 10)):.1f} %, "
      f"X >= 10 — {100 * np.mean(cooc_value >= 10):.2f} %")

## 11 · Word2Vec (skip-gram) на тих самих даних

Щоб порівняння було чесним, skip-gram теж пишемо самі — та сама розмірність,
те саме вікно, ті самі три зерна. Це скорочений код теми 12: пара «слово-сусід»
проти пʼятьох випадкових слів, узятих із частотного розподілу в степені 0.75.

In [ ]:
NEGATIVE = 5
noise_distribution = np.array([word_counts[w] for w in vocabulary], dtype=np.float64) ** 0.75
noise_distribution /= noise_distribution.sum()
noise_cumulative = np.cumsum(noise_distribution)


def train_skipgram(seed, dim=64, epochs=1, batch=2048, lr=0.05, subword_matrix=None):
    """Skip-gram із негативним семплюванням.
    subword_matrix=None — звичайний Word2Vec: у кожного слова свій вектор.
    Інакше вектор слова збирається з рядка цієї матриці (це і є FastText)."""
    rng = np.random.default_rng(seed)
    size = len(vocabulary) if subword_matrix is None else subword_matrix.shape[1]
    inputs = ((rng.random((size, dim), dtype=np.float32) - 0.5) / dim).astype(np.float32)
    outputs = np.zeros((len(vocabulary), dim), dtype=np.float32)
    n = len(pair_center)
    seen, total = 0, epochs * n
    for _ in range(epochs):
        order = rng.permutation(n)
        for start in range(0, n, batch):
            k = order[start:start + batch]
            m = len(k)
            rate = lr * max(1e-4, 1 - seen / total)      # крок лінійно спадає до нуля
            seen += m
            center, context = pair_center[k], pair_context[k]
            if subword_matrix is None:
                hidden = inputs[center]
            else:
                rows = subword_matrix[center]            # розріджений рядок: слово + його n-грами
                hidden = (rows @ inputs).astype(np.float32)
            negative = np.searchsorted(noise_cumulative,
                                       rng.random(m * NEGATIVE)).astype(np.int32).reshape(m, NEGATIVE)
            targets = np.concatenate([context[:, None], negative], axis=1)
            target_vectors = outputs[targets]
            score = np.einsum('md,mkd->mk', hidden, target_vectors)
            probability = 1 / (1 + np.exp(-np.clip(score, -6, 6)))
            # мітка 1 для справжнього сусіда і 0 для пʼятьох випадкових
            gradient = -probability * rate
            gradient[:, 0] += rate
            hidden_gradient = np.einsum('mk,mkd->md', gradient, target_vectors).astype(np.float32)
            output_gradient = (gradient[:, :, None] * hidden[:, None, :]).reshape(-1, dim)
            scatter_add(outputs, targets.ravel(), output_gradient.astype(np.float32))
            if subword_matrix is None:
                scatter_add(inputs, center, hidden_gradient)
            else:
                inputs += (rows.T @ hidden_gradient).astype(np.float32)
    # повертаємо обидві матриці: центральну (вектор слова) і контекстну (вектор
    # сусіда). У коді й у статтях їх звуть вхідною та вихідною — це та сама пара.
    return inputs, outputs


w2v_scores, w2v_times, w2v_models, w2v_outputs = [], [], [], []
for seed in SEEDS:
    started = time.process_time()
    vectors, outputs = train_skipgram(seed)
    spent = time.process_time() - started
    mrr, top1, _ = retrieval_score(vectors)
    w2v_scores.append(mrr)
    w2v_times.append(spent)
    w2v_models.append(vectors)
    w2v_outputs.append(outputs)
    print(f"зерно {seed}: MRR {mrr:.4f}, перший з першого разу {100 * top1:.1f} %, "
          f"навчання {spent:.1f} с")

print()
print(f"Word2Vec: MRR {np.mean(w2v_scores):.4f} ± {np.std(w2v_scores):.4f}, "
      f"навчання {np.mean(w2v_times):.1f} с")
print()
for word in ('файл', 'помилка', 'вікно', 'мережі'):
    print(f"   {word:<9} -> {' · '.join(nearest(w2v_models[0], word))}")

## 12 · Чи видно різницю між GloVe і Word2Vec

Тепер головне питання розділу: різниця між методами більша за розкид між зернами
чи ні? Якщо ні — казати «цей метод кращий» не можна.

In [ ]:
gap = np.mean(w2v_scores) - np.mean(glove_scores)
spread = max(np.std(w2v_scores), np.std(glove_scores))
print(f"{'метод':<12} {'MRR':>8} {'розкид':>9} {'навчання':>10}")
print(f"{'GloVe':<12} {np.mean(glove_scores):>8.4f} {np.std(glove_scores):>9.4f} "
      f"{np.mean(glove_times):>9.1f} с")
print(f"{'Word2Vec':<12} {np.mean(w2v_scores):>8.4f} {np.std(w2v_scores):>9.4f} "
      f"{np.mean(w2v_times):>9.1f} с")
print()
print(f"різниця середніх: {gap:+.4f}, найбільший розкид: {spread:.4f}")
print("купи по зернах:", [round(s, 4) for s in sorted(glove_scores)],
      "проти", [round(s, 4) for s in sorted(w2v_scores)])
if abs(gap) > 2 * spread:
    print("ВИСНОВОК: різниця більша за подвоєний розкид — вона є")
else:
    print("ВИСНОВОК: різниця менша за подвоєний розкид — на цих даних методи нерозрізненні")

### Проміжна зупинка: який бік розкладу ми беремо

І GloVe, і skip-gram дають на виході **дві** матриці на той самий словник:
**центральну** (вектор слова, коли воно посередині) і **контекстну** (вектор
того самого слова, коли воно сусід). У коді й у статтях цю пару звуть вхідною та
вихідною матрицею (input / output embeddings) — це ті самі дві матриці. Питання
«яку з них вважати вектором слова» виглядає технічним, а насправді міняє
результат сильніше за все, що ми міряємо далі.

Моделі вже навчені, тож перевірка безкоштовна: перерахуємо MRR для кожного вибору.

І одразу назвемо рішення, бо воно неочевидне. Автори GloVe радять **суму**. Але
FastText уміє зібрати з субслів лише **центральну** матрицю — у слова, якого
модель ніколи не бачила, контекстного рядка не існує. Якщо взяти суму в GloVe й
Word2Vec, а у FastText — тільки центральну частину, то замір порівнюватиме не
субслова, а деталь реалізації. Тому в усій темі всі три моделі беруть
**центральну матрицю**.

In [ ]:
# Кожна з двох моделей дає ДВІ матриці, і вибір між ними — не дрібниця.
# Моделі вже навчені, тож цей замір коштує лише перерахунку MRR.
print(f"{'модель':<10} {'який бік беремо':<28} {'MRR':>8} {'розкид':>9}")
side_scores = {}
for name, sides in (("GloVe", ('main', 'ctx', 'sum')), ("Word2Vec", ('in', 'out', 'sum'))):
    for side in sides:
        values = []
        for k, seed in enumerate(SEEDS):
            if name == "GloVe":
                vectors = glove_vectors(glove_models[k], side)
            elif side == 'in':
                vectors = w2v_models[k]
            elif side == 'out':
                vectors = w2v_outputs[k]
            else:
                vectors = w2v_models[k] + w2v_outputs[k]
            values.append(retrieval_score(vectors)[0])
        side_scores[(name, side)] = values
        label = {'main': 'центральна матриця', 'ctx': 'контекстна матриця',
                 'in': 'центральна матриця', 'out': 'контекстна матриця',
                 'sum': 'сума двох матриць'}[side]
        print(f"{name:<10} {label:<28} {np.mean(values):>8.4f} {np.std(values):>9.4f}")

best_glove = max(('main', 'ctx', 'sum'), key=lambda s: np.mean(side_scores[("GloVe", s)]))
best_w2v = max(('in', 'out', 'sum'), key=lambda s: np.mean(side_scores[("Word2Vec", s)]))
print()
print(f"найкращий бік у GloVe: {best_glove}, у Word2Vec: {best_w2v}")
spread_here = max(np.std(v) for v in side_scores.values())
worst_gap = (max(np.mean(v) for v in side_scores.values())
             - min(np.mean(v) for v in side_scores.values()))
print(f"розкид між боками: {worst_gap:.4f} при найбільшому розкиді по зернах {spread_here:.4f}")
print(f"різниця між самими методами (центральні матриці): "
      f"{np.mean(w2v_scores) - np.mean(glove_scores):+.4f}")
print()
print("⚠️  FastText уміє зібрати з субслів лише ЦЕНТРАЛЬНУ матрицю: у слова поза")
print("    словником контекстного рядка немає взагалі. Тому далі всі три моделі")
print("    беруть один і той самий бік — центральну матрицю.")

## 13 · Субслова: слово як мішок своїх n-грам

Тепер друга половина теми. Обидві моделі вище мають одну спільну ваду: слово,
якого не було в словнику, для них не існує взагалі. А ми щойно порахували, що
таких словоформ у нашому корпусі більшість.

FastText лікує це так: слово обгортають кутовими дужками й ріжуть на всі
підрядки довжиною від трьох до пʼяти літер. Вектор слова — **середнє** векторів
його шматків і його власного вектора. Дужки потрібні, щоб «кінець слова» був
видимий: `ція>` — це закінчення, а `ція` посеред слова — просто склад.

In [ ]:
NGRAM_MIN, NGRAM_MAX = 3, 5

def subwords(word):
    """Усі підрядки довжиною 3-5 літер, разом із межами слова."""
    marked = '<' + word + '>'
    pieces = []
    for size in range(NGRAM_MIN, NGRAM_MAX + 1):
        for start in range(len(marked) - size + 1):
            pieces.append(marked[start:start + size])
    return pieces

example = 'кореневому'
print(f"слово «{example}» ({len(subwords(example))} субслів):")
print("  ", ' '.join(subwords(example)[:14]), "…")
print()
subword_index = {}
rows, cols, values = [], [], []
for i, word in enumerate(vocabulary):
    parts = [i]                                   # власний вектор слова
    for piece in subwords(word):
        if piece not in subword_index:
            subword_index[piece] = len(vocabulary) + len(subword_index)
        parts.append(subword_index[piece])
    share = 1.0 / len(parts)                      # середнє, а не сума
    for part in parts:
        rows.append(i)
        cols.append(part)
        values.append(share)

PARAMS = len(vocabulary) + len(subword_index)
subword_matrix = sparse.csr_matrix((np.array(values, dtype=np.float32), (rows, cols)),
                                   shape=(len(vocabulary), PARAMS))
known_subwords = set(subword_index)

print(f"різних субслів у словнику: {len(subword_index)}")
print(f"рядків у таблиці векторів: {PARAMS} проти {len(vocabulary)} у Word2Vec "
      f"— у {PARAMS / len(vocabulary):.2f} раза більше")
print(f"субслів на слово в середньому: {len(cols) / len(vocabulary):.1f}")

In [ ]:
def assemble_vector(word, table):
    """Вектор будь-якого слова — навіть того, якого модель не бачила."""
    parts = []
    if word in word_index:
        parts.append(word_index[word])
    for piece in subwords(word):
        if piece in subword_index:
            parts.append(subword_index[piece])
    if not parts:
        return None
    return table[parts].mean(axis=0)

# перевірка: збірка «руками» має дати те саме, що множення на розріджену матрицю
probe_table = np.random.default_rng(3).random((PARAMS, 8)).astype(np.float32)
by_matrix = (subword_matrix @ probe_table)[word_index['файл']]
by_hand = assemble_vector('файл', probe_table)
assert np.allclose(by_matrix, by_hand, atol=1e-5), "збірка вектора розійшлася!"
print("✅ збірка з субслів збігається з множенням на матрицю")
print("вектор слова, якого в словнику немає, теж збирається:",
      assemble_vector('кореневому', probe_table) is not None)

## 14 · FastText: те саме навчання, інший вхід

Ніякого окремого алгоритму тут немає. Це той самий skip-gram, тільки на вході
замість одного рядка таблиці стоїть середнє рядків слова та його субслів. Ми
вже написали цю гілку у функції `train_skipgram` — досить передати матрицю.

In [ ]:
ft_scores, ft_times, ft_models, ft_outputs = [], [], [], []
for seed in SEEDS:
    started = time.process_time()
    table, outputs = train_skipgram(seed, subword_matrix=subword_matrix)
    spent = time.process_time() - started
    vectors = (subword_matrix @ table).astype(np.float32)
    mrr, top1, _ = retrieval_score(vectors)
    ft_scores.append(mrr)
    ft_times.append(spent)
    ft_models.append(table)
    ft_outputs.append(outputs)
    print(f"зерно {seed}: MRR {mrr:.4f}, перший з першого разу {100 * top1:.1f} %, "
          f"навчання {spent:.1f} с")

ft_vectors = (subword_matrix @ ft_models[0]).astype(np.float32)
print()
print(f"FastText: MRR {np.mean(ft_scores):.4f} ± {np.std(ft_scores):.4f}, "
      f"навчання {np.mean(ft_times):.1f} с")
print(f"проти Word2Vec: MRR {np.mean(w2v_scores):.4f} ± {np.std(w2v_scores):.4f}, "
      f"навчання {np.mean(w2v_times):.1f} с")
print(f"ціна субслів: параметрів у {PARAMS / len(vocabulary):.2f} раза більше, "
      f"часу — у {np.mean(ft_times) / np.mean(w2v_times):.1f} раза")
print()
for word in ('файл', 'помилка', 'вікно', 'мережі'):
    print(f"   {word:<9} -> {' · '.join(nearest(ft_vectors, word))}")

## 15 · Оптимістична оцінка: скільки OOV «впізнають» субслова

Тепер головне питання теми. Модель не знає більшості словоформ — чи рятують
субслова?

Найпростіший спосіб відповісти: узяти невідоме слово, порізати на субслова й
подивитись, скільки з них уже трапилось у відомих словах. Саме так зазвичай і
міряють, і саме тут ховається пастка.

In [ ]:
started = time.process_time()
seen_ngrams = set()
for word in vocabulary:
    seen_ngrams.update(subwords(word))

known_share = {}
for word in oov_words:
    pieces = subwords(word)
    if not pieces:
        continue
    known_share[word] = sum(1 for piece in pieces if piece in seen_ngrams) / len(pieces)

shares = np.array(list(known_share.values()))
recognised = int((shares > 0.5).sum())
print(f"словоформ поза словником: {len(oov_words)}")
print(f"у яких понад половина субслів уже відома: {recognised} "
      f"({100 * recognised / len(known_share):.1f} %)")
print(f"середня частка відомих субслів: {shares.mean():.4f}")
print(f"зайняло {time.process_time() - started:.1f} с процесорного часу")

## 16 · Чесна оцінка: а чи справді відома лема

«Понад половина субслів відома» звучить як «модель майже все впізнає». Перевіримо
це прямо: візьмемо лему невідомого слова (`pymorphy3` із теми 03) і подивимось,
чи є серед **відомих** слів хоч одна форма тієї самої леми. Якщо є — модель
справді має до чого причепити нове слово. Якщо ні — субслова впізнали щось інше.

Поруч поставимо ще простіший спосіб, знайомий із теми 03: відрізати від слова
чотири останні літери (але лишити щонайменше три) і подивитись, чи дасть якесь
відоме слово ту саму основу.

In [ ]:
started = time.process_time()
morph = pymorphy3.MorphAnalyzer(lang='uk')

lemma_of_known = {}
for word in vocabulary:
    lemma_of_known[word] = morph.parse(word)[0].normal_form
known_by_lemma = collections.defaultdict(list)
for word, lemma in lemma_of_known.items():
    known_by_lemma[lemma].append(word)

lemma_of_oov = {}
for word in oov_words:
    lemma_of_oov[word] = morph.parse(word)[0].normal_form
with_known_lemma = [word for word in oov_words if lemma_of_oov[word] in known_by_lemma]


def simple_stem(word, cut=4, keep=3):
    """Основа з теми 03: відрізати чотири літери, але не коротше за три."""
    return word[:-cut] if len(word) - cut >= keep else word

stems_of_known = {simple_stem(word) for word in vocabulary}
with_known_stem = [word for word in oov_words if simple_stem(word) in stems_of_known]

print(f"{'спосіб оцінки':<42} {'впізнано':>10} {'частка':>9}")
print(f"{'понад половина субслів відома':<42} {recognised:>10} "
      f"{100 * recognised / len(oov_words):>8.1f} %")
print(f"{'лема справді є серед відомих слів':<42} {len(with_known_lemma):>10} "
      f"{100 * len(with_known_lemma) / len(oov_words):>8.1f} %")
print(f"{'проста основа (−4) є серед відомих':<42} {len(with_known_stem):>10} "
      f"{100 * len(with_known_stem) / len(oov_words):>8.1f} %")
print()
print(f"оптимістична оцінка більша за чесну у "
      f"{recognised / max(1, len(with_known_lemma)):.1f} раза")
print(f"лематизація зайняла {time.process_time() - started:.1f} с")
print()
print("приклади невідомих слів, у яких лема таки відома:")
for word in ('конфігурацією', 'неправильною', 'необхідного', 'кореневому', 'перемикатися'):
    if word in lemma_of_oov:
        mates = known_by_lemma.get(lemma_of_oov[word], [])
        print(f"   {word:<16} лема «{lemma_of_oov[word]}», відомі форми: "
              f"{' · '.join(mates[:3]) if mates else '—'}")

## 17 · Чому оцінки розходяться: спільні субслова — це закінчення

Різниця майже втричі мусить мати причину, і вона проста. Розділимо субслова
слова на три групи: ті, що торкаються **початку** (містять `<`), ті, що
торкаються **кінця** (містять `>`), і серединні. Далі порівняємо дві купи слів:
ті, у яких лема відома, і ті, у яких ні.

In [ ]:
lemma_known_flag = {word: (lemma_of_oov[word] in known_by_lemma) for word in oov_words}

def group_shares(word):
    """Частка відомих субслів окремо на початку, в кінці й посередині слова."""
    pieces = subwords(word)
    head = [p for p in pieces if p.startswith('<')]
    tail = [p for p in pieces if p.endswith('>')]
    middle = [p for p in pieces if not p.startswith('<') and not p.endswith('>')]
    out = []
    for group in (head, tail, middle):
        out.append(sum(1 for p in group if p in seen_ngrams) / len(group) if group else np.nan)
    return out

head_share, tail_share, middle_share, flags = [], [], [], []
for word in oov_words:
    h, t, m = group_shares(word)
    head_share.append(h)
    tail_share.append(t)
    middle_share.append(m)
    flags.append(lemma_known_flag[word])
head_share = np.array(head_share)
tail_share = np.array(tail_share)
middle_share = np.array(middle_share)
flags = np.array(flags)

print(f"{'':<22} {'лема відома':>13} {'лема невідома':>15}")
print(f"{'початок слова (корінь)':<22} {np.nanmean(head_share[flags]):>13.4f} "
      f"{np.nanmean(head_share[~flags]):>15.4f}")
print(f"{'кінець слова':<22} {np.nanmean(tail_share[flags]):>13.4f} "
      f"{np.nanmean(tail_share[~flags]):>15.4f}")
print(f"{'середина':<22} {np.nanmean(middle_share[flags]):>13.4f} "
      f"{np.nanmean(middle_share[~flags]):>15.4f}")
print()
all_share = np.array([known_share.get(w, 0.0) for w in oov_words])
print(f"частка відомих субслів загалом: лема відома {all_share[flags].mean():.4f}, "
      f"лема невідома {all_share[~flags].mean():.4f}")
print(f"«понад половина відома»: серед слів із відомою лемою "
      f"{100 * np.mean(all_share[flags] > 0.5):.1f} %, серед решти "
      f"{100 * np.mean(all_share[~flags] > 0.5):.1f} %")

Ось і відповідь. Субслова початку слова — тобто ті, що несуть корінь, — справді
розрізняють дві купи. А субслова кінця відомі майже однаково часто в обох, бо
закінчень в українській мові десятки, і всі вони давно трапились. Тому загальна
частка відомих субслів так слабо відрізняє «модель знає це слово» від «модель
знає, що воно українське».

Ця клітинка друкує таблицю для фігури лекції: як міняється «впізнано» і його
чесна точність, якщо посувати поріг.

In [ ]:
print(f"{'поріг':>7} {'«впізнано»':>12} {'із них лема відома':>20} {'точність':>10}")
for threshold in (0.0, 0.25, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0):
    picked = all_share > threshold
    correct = int((picked & flags).sum())
    total = int(picked.sum())
    precision = correct / total if total else 0.0
    print(f"{threshold:>7.2f} {total:>12} {correct:>20} {100 * precision:>9.1f} %")

## 18 · Перший чесний замір: знайти відоме споріднене слово

Перекриття n-грам — це не задача, а проміжна величина. Поставимо задачу:
для невідомого слова знайти в словнику **споріднене** слово, тобто форму тієї
самої леми. Правильну відповідь ми знаємо від `pymorphy3`, тож можна порахувати,
хто скільки вгадав.

Порівнюємо три способи:

- **n-грами без навчання** — просто найбільше перекриття субслів;
- **FastText** — зібрати вектор із субслів і взяти найближче слово за косинусом;
- **основа −4** — узяти найчастіше відоме слово з тією самою основою.

Якщо навчені субслова не обходять просте порівняння рядків, то платити за них
памʼяттю й часом немає сенсу.

In [ ]:
SAMPLE = 1200

# розріджена матриця «слово × його субслова» для порівняння без навчання
def ngram_rows(words):
    r, c = [], []
    for i, word in enumerate(words):
        for piece in set(subwords(word)):
            if piece in subword_index:
                r.append(i)
                c.append(subword_index[piece] - len(vocabulary))
    matrix = sparse.csr_matrix((np.ones(len(r), dtype=np.float32), (r, c)),
                               shape=(len(words), len(subword_index)))
    length = np.sqrt(np.asarray(matrix.multiply(matrix).sum(1)).ravel())
    length[length == 0] = 1
    return sparse.diags(1 / length) @ matrix

known_ngram_rows = ngram_rows(vocabulary)
stem_to_words = collections.defaultdict(list)
for word in vocabulary:
    stem_to_words[simple_stem(word)].append(word)


def relatives(word):
    """Відомі слова, що мають ту саму лему, — правильні відповіді."""
    return set(known_by_lemma.get(lemma_of_oov[word], []))


ngram_hits, fasttext_hits, stem_hits, chance = [], [], [], []
for seed in SEEDS:
    rng = np.random.default_rng(seed)
    picked = [with_known_lemma[i] for i in
              rng.choice(len(with_known_lemma), min(SAMPLE, len(with_known_lemma)), replace=False)]
    answers = [relatives(word) for word in picked]

    # 1. чисте перекриття n-грам, жодного навчання
    similarity = np.asarray((ngram_rows(picked) @ known_ngram_rows.T).todense())
    ngram_hits.append(np.mean([vocabulary[int(np.argmax(similarity[i]))] in answers[i]
                               for i in range(len(picked))]))

    # 2. навчені субслова: збираємо вектор і шукаємо найближчий
    table = ft_models[SEEDS.index(seed)]
    word_vectors = (subword_matrix @ table).astype(np.float32)
    word_vectors = word_vectors / np.linalg.norm(word_vectors, axis=1, keepdims=True)
    guesses = np.array([assemble_vector(word, table) for word in picked], dtype=np.float32)
    guesses = guesses / np.linalg.norm(guesses, axis=1, keepdims=True)
    scores = guesses @ word_vectors.T
    fasttext_hits.append(np.mean([vocabulary[int(np.argmax(scores[i]))] in answers[i]
                                  for i in range(len(picked))]))

    # 3. проста основа
    hit = 0
    for word, answer in zip(picked, answers):
        candidates = stem_to_words.get(simple_stem(word), [])
        if candidates:
            best = max(candidates, key=lambda c: word_counts[c])
            hit += best in answer
    stem_hits.append(hit / len(picked))

    # 4. випадкове слово словника — щоб бачити, з чим порівнюємо
    chance.append(np.mean([len(answer) / len(vocabulary) for answer in answers]))

print(f"{'спосіб':<30} {'вгадав':>9} {'розкид':>9}")
for name, values in (("n-грами без навчання", ngram_hits),
                     ("FastText (навчені субслова)", fasttext_hits),
                     ("основа −4", stem_hits),
                     ("випадкове слово словника", chance)):
    print(f"{name:<30} {100 * np.mean(values):>8.1f} % {100 * np.std(values):>8.2f} %")

## 19 · Другий чесний замір: та сама задача пошуку, але з невідомими словами

Перший замір показує, чи вміють субслова знаходити родичів. Другий — чи допомагає
це **на задачі**, заради якої все й робиться.

Беремо ті самі пари «те саме іншими словами», але лише ті, де є хоч одне слово
поза словником. І порівнюємо три способи скласти вектор документа:

- **Word2Vec** — невідомі слова просто пропускаємо, іншого виходу немає;
- **FastText, невідомі пропущено** — та сама модель, але ми свідомо не збираємо
  вектор із субслів;
- **FastText, невідомі зібрано** — усе як має бути.

Третій рядок відрізняється від другого **тільки** реконструкцією. Це і є чиста
ціна питання.

In [ ]:
with_oov = [i for i, (a, b) in enumerate(pairs)
            if any(w not in known_words for w in tokenize(a.lower()) + tokenize(b.lower()))]
print(f"пар із невідомими словами: {len(with_oov)} із {len(pairs)}")

w2v_on_oov, ft_dropped, ft_rebuilt = [], [], []
for k, seed in enumerate(SEEDS):
    table = ft_models[k]
    vectors = (subword_matrix @ table).astype(np.float32)
    def rebuild(word, table=table):
        return assemble_vector(word, table)
    w2v_on_oov.append(retrieval_score(w2v_models[k], subset=with_oov)[0])
    ft_dropped.append(retrieval_score(vectors, subset=with_oov)[0])
    ft_rebuilt.append(retrieval_score(vectors, unknown_vector=rebuild, subset=with_oov)[0])

print()
print(f"{'спосіб':<34} {'MRR':>8} {'розкид':>9}")
for name, values in (("Word2Vec, невідомі пропущено", w2v_on_oov),
                     ("FastText, невідомі пропущено", ft_dropped),
                     ("FastText, невідомі зібрано", ft_rebuilt)):
    print(f"{name:<34} {np.mean(values):>8.4f} {np.std(values):>9.4f}")
print()
gain = np.mean(ft_rebuilt) - np.mean(ft_dropped)
print(f"чистий внесок реконструкції: {gain:+.4f} "
      f"(розкид {max(np.std(ft_rebuilt), np.std(ft_dropped)):.4f})")
print(f"відставання FastText від Word2Vec на тих самих парах: "
      f"{np.mean(ft_rebuilt) - np.mean(w2v_on_oov):+.4f}")

## 20 · Де субслова шкодять

У субслів є зворотний бік, і він теж вимірний. Два слова з однаковими літерами
можуть не мати нічого спільного за змістом — а FastText бачить у них спільні
шматки й підтягує їхні вектори один до одного.

Порахуємо це на **всіх** парах слів словника — їх понад вісімнадцять мільйонів.
Пари ділимо на дві купи: ті, у яких **та сама лема** (тобто схожість написання
чесна), і ті, у яких **леми різні** (схожість випадкова). Далі дивимось, що про
кожну купу кажуть обидві моделі.

In [ ]:
lemma_codes = {}
lemma_code = np.array([lemma_codes.setdefault(lemma_of_known[word], len(lemma_codes))
                       for word in vocabulary])

w2v_unit = w2v_models[0] / np.linalg.norm(w2v_models[0], axis=1, keepdims=True)
ft_unit = ft_vectors / np.linalg.norm(ft_vectors, axis=1, keepdims=True)

EDGES = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 1.01]
stats = {(b, same): [0, 0.0, 0.0] for b in range(len(EDGES) - 1) for same in (True, False)}
examples = []

CHUNK = 500
started = time.process_time()
for begin in range(0, len(vocabulary), CHUNK):
    stop = min(begin + CHUNK, len(vocabulary))
    overlap = np.asarray((known_ngram_rows[begin:stop] @ known_ngram_rows.T).todense())
    w2v_here = w2v_unit[begin:stop] @ w2v_unit.T
    ft_here = ft_unit[begin:stop] @ ft_unit.T
    # кожну пару беремо один раз: лише праворуч від діагоналі
    upper = np.arange(len(vocabulary))[None, :] > np.arange(begin, stop)[:, None]
    same_lemma = lemma_code[begin:stop][:, None] == lemma_code[None, :]
    for b in range(len(EDGES) - 1):
        in_bin = upper & (overlap >= EDGES[b]) & (overlap < EDGES[b + 1])
        for same in (True, False):
            picked = in_bin & (same_lemma if same else ~same_lemma)
            cell = stats[(b, same)]
            cell[0] += int(picked.sum())
            cell[1] += float(w2v_here[picked].sum())
            cell[2] += float(ft_here[picked].sum())
    strange = upper & (~same_lemma) & (overlap > 0.45)
    for i, j in zip(*np.nonzero(strange)):
        examples.append((float(w2v_here[i, j] - ft_here[i, j]), vocabulary[begin + i],
                         vocabulary[j], float(overlap[i, j]),
                         float(w2v_here[i, j]), float(ft_here[i, j])))

print(f"{'перекриття субслів':>20} {'та сама лема':>14} {'word2vec':>10} {'fasttext':>10} "
      f"{'різні леми':>12} {'word2vec':>10} {'fasttext':>10}")
for b in range(len(EDGES) - 1):
    same_cell, other_cell = stats[(b, True)], stats[(b, False)]
    label = f"{EDGES[b]:.1f}-{min(EDGES[b + 1], 1.0):.1f}"
    same_text = (f"{same_cell[1] / same_cell[0]:>10.4f} {same_cell[2] / same_cell[0]:>10.4f}"
                 if same_cell[0] else f"{'—':>10} {'—':>10}")
    other_text = (f"{other_cell[1] / other_cell[0]:>10.4f} {other_cell[2] / other_cell[0]:>10.4f}"
                  if other_cell[0] else f"{'—':>10} {'—':>10}")
    print(f"{label:>20} {same_cell[0]:>14} {same_text} {other_cell[0]:>12} {other_text}")
print(f"пораховано за {time.process_time() - started:.1f} с процесорного часу")

In [ ]:
high_same = sum(stats[(b, True)][0] for b in range(4, 7))
high_other = sum(stats[(b, False)][0] for b in range(4, 7))
w2v_same = sum(stats[(b, True)][1] for b in range(4, 7)) / max(1, high_same)
w2v_other = sum(stats[(b, False)][1] for b in range(4, 7)) / max(1, high_other)
ft_same = sum(stats[(b, True)][2] for b in range(4, 7)) / max(1, high_same)
ft_other = sum(stats[(b, False)][2] for b in range(4, 7)) / max(1, high_other)
print("пари з перекриттям субслів понад 0.4:")
print(f"   та сама лема ({high_same} пар):  word2vec {w2v_same:.4f}, fasttext {ft_same:.4f}")
print(f"   різні леми  ({high_other} пар):  word2vec {w2v_other:.4f}, fasttext {ft_other:.4f}")
print(f"   word2vec розрізняє купи на {w2v_same - w2v_other:.4f}, "
      f"fasttext — лише на {ft_same - ft_other:.4f}")
print()
print("різні леми, але FastText вважає їх майже одним словом:")
examples.sort()
for difference, first, second, share, w2v_score, ft_score in examples[:8]:
    print(f"   {first:<16} {second:<16} субслова {share:.3f}   "
          f"word2vec {w2v_score:+.3f}   fasttext {ft_score:+.3f}")
print()
print("· накопичено за порогом перекриття (для фігури лекції)")
print(f"{'поріг':>6} {'та сама лема':>13} {'w2v':>8} {'ft':>8} "
      f"{'різні леми':>11} {'w2v':>8} {'ft':>8} {'розрив w2v':>11} {'розрив ft':>10}")
for b in range(len(EDGES) - 1):
    same_n = sum(stats[(k, True)][0] for k in range(b, len(EDGES) - 1))
    other_n = sum(stats[(k, False)][0] for k in range(b, len(EDGES) - 1))
    same_w = sum(stats[(k, True)][1] for k in range(b, len(EDGES) - 1)) / max(1, same_n)
    other_w = sum(stats[(k, False)][1] for k in range(b, len(EDGES) - 1)) / max(1, other_n)
    same_f = sum(stats[(k, True)][2] for k in range(b, len(EDGES) - 1)) / max(1, same_n)
    other_f = sum(stats[(k, False)][2] for k in range(b, len(EDGES) - 1)) / max(1, other_n)
    print(f"{EDGES[b]:>6.1f} {same_n:>13} {same_w:>8.4f} {same_f:>8.4f} "
          f"{other_n:>11} {other_w:>8.4f} {other_f:>8.4f} "
          f"{same_w - other_w:>11.4f} {same_f - other_f:>10.4f}")

Є й друга, тонша причина, чому зібраний вектор слабший, ніж здається. Він
усереднює понад два десятки шматків, більшість із яких — звичайні українські
закінчення. Що більше доданків, то ближче середнє до центру всієї хмари векторів,
тобто до «просто якогось українського слова».

Заміряємо це прямо: наскільки кожен вид вектора близький до центру хмари.

In [ ]:
centre = ft_vectors.mean(axis=0)
centre = centre / np.linalg.norm(centre)
known_unit = ft_vectors / np.linalg.norm(ft_vectors, axis=1, keepdims=True)
sample_oov = with_known_lemma[:2000]
built = np.array([assemble_vector(word, ft_models[0]) for word in sample_oov], dtype=np.float32)
built = built / np.linalg.norm(built, axis=1, keepdims=True)
print(f"косинус із центром хмари: відомі слова {float(np.mean(known_unit @ centre)):.4f}, "
      f"зібрані з субслів {float(np.mean(built @ centre)):.4f}")
print(f"довжина вектора до нормування: відомі слова "
      f"{float(np.mean(np.linalg.norm(ft_vectors, axis=1))):.4f}, зібрані "
      f"{float(np.mean(np.linalg.norm(np.array([assemble_vector(w, ft_models[0]) for w in sample_oov]), axis=1))):.4f}")

## 21 · Ціна субслів

Останнє, що треба знати перед вибором: скільки все це коштує.

In [ ]:
print(f"{'':<34} {'Word2Vec':>12} {'FastText':>12}")
print(f"{'рядків у таблиці векторів':<34} {len(vocabulary):>12} {PARAMS:>12}")
print(f"{'чисел у моделі (64 виміри)':<34} {64 * len(vocabulary):>12} {64 * PARAMS:>12}")
print(f"{'памʼять float32, МБ':<34} {64 * len(vocabulary) * 4 / 1e6:>12.2f} "
      f"{64 * PARAMS * 4 / 1e6:>12.2f}")
print(f"{'навчання, с процесорних':<34} {np.mean(w2v_times):>12.1f} {np.mean(ft_times):>12.1f}")
print(f"{'MRR на парах-синонімах':<34} {np.mean(w2v_scores):>12.4f} {np.mean(ft_scores):>12.4f}")
print()
print(f"параметрів більше у {PARAMS / len(vocabulary):.2f} раза, "
      f"часу — у {np.mean(ft_times) / np.mean(w2v_times):.1f} раза, "
      f"якість на відомих словах — {np.mean(ft_scores) - np.mean(w2v_scores):+.4f}")

## 22 · Дані для фігур лекції

Фігури лекції рахують усе в браузері, але числа в них — справжні, з цього
зошита. Ця клітинка друкує саме ті таблиці, які туди зашиті.

In [ ]:
print("· частотний спектр: скільки словоформ трапилось рівно k разів")
spectrum = collections.Counter(word_counts.values())
print("   " + " ".join(f"{k}:{spectrum.get(k, 0)}" for k in range(1, 22)))
print(f"   разом словоформ {len(word_counts)}, слововживань {total_tokens}")

print()
print("· клітинки таблиці співвіднесень різної величини — для фігури про формулу")
for target in (3000.0, 300.0, 30.0, 3.0, 0.5):
    k = int(np.argmin(np.abs(cooc_value - target)))
    print(f"   X({vocabulary[cooc_left[k]]}, {vocabulary[cooc_right[k]]}) = "
          f"{cooc_value[k]:.4f}, log X = {math.log(float(cooc_value[k])):.4f}")

print()
print("· куди вагова функція вкладає зусилля моделі (частка суми f(X) по купах)")
low = cooc_value < 1
mid = (cooc_value >= 1) & (cooc_value < 10)
high = cooc_value >= 10
print(f"{'x_max':>6} {'alpha':>6} {'обрізання':>10} {'X < 1':>9} {'1..10':>9} {'X >= 10':>9}")
for x_max in (2.0, 5.0, 10.0, 20.0, 50.0, 100.0):
    for alpha in (0.25, 0.5, 0.75, 1.0):
        for clipped in (True, False):
            raw = (cooc_value / x_max) ** alpha
            w = np.minimum(1.0, raw) if clipped else raw
            whole = float(w.sum())
            print(f"{x_max:>6.0f} {alpha:>6.2f} {'так' if clipped else 'ні':>10} "
                  f"{100 * float(w[low].sum()) / whole:>8.2f} % "
                  f"{100 * float(w[mid].sum()) / whole:>8.2f} % "
                  f"{100 * float(w[high].sum()) / whole:>8.2f} %")

print()
print("· субслова пʼятьох слів по довжинах: скільки з них уже відомі моделі")
FIGURE_WORDS = ('конфігурацією', 'неправильною', 'кореневому', 'перемикатися', 'необхідного')
ngrams_by_length = {}
for size in range(3, 7):
    seen_of_size = set()
    for word in vocabulary:
        marked = '<' + word + '>'
        for start in range(len(marked) - size + 1):
            seen_of_size.add(marked[start:start + size])
    ngrams_by_length[size] = seen_of_size

for word in FIGURE_WORDS:
    marked = '<' + word + '>'
    parts = []
    for size in range(3, 7):
        pieces = [marked[i:i + size] for i in range(len(marked) - size + 1)]
        hits = sum(1 for p in pieces if p in ngrams_by_length[size])
        parts.append(f"n={size}: {hits}/{len(pieces)}")
    print(f"   {word:<15} " + "  ".join(parts))

print()
print("· плитки для фігури (1 — субслово вже відоме, 0 — ні)")
for word in FIGURE_WORDS:
    marked = '<' + word + '>'
    for size in range(3, 7):
        pieces = [marked[i:i + size] for i in range(len(marked) - size + 1)]
        flags_here = ''.join('1' if p in ngrams_by_length[size] else '0' for p in pieces)
        print(f"   {word:<15} n={size} {flags_here}")

print()
print("· поріг «впізнано» дрібним кроком: скільки слів і скільки з них із відомою лемою")
for step in range(21):
    threshold = step / 20
    picked = all_share > threshold
    total = int(picked.sum())
    correct = int((picked & flags).sum())
    print(f"   поріг {threshold:.2f}: впізнано {total:>5}, лема відома {correct:>5}, "
          f"точність {100 * correct / total if total else 0:5.1f} %")

print()
print("· гістограма: частка відомих субслів × чи відома лема")
edges = np.linspace(0, 1, 21)
for low_edge, high_edge in zip(edges[:-1], edges[1:]):
    inside = (all_share >= low_edge) & (all_share < high_edge + (1e-9 if high_edge == 1 else 0))
    print(f"   [{low_edge:.2f}, {high_edge:.2f}): всього {int(inside.sum()):>5}, "
          f"лема відома {int((inside & flags).sum()):>5}")

print()
print("· внесок реконструкції по зернах:", [round(x, 4) for x in ft_rebuilt],
      "проти", [round(x, 4) for x in ft_dropped])

## 23 · Що з цього виходить

**01.** При порозі `min_count = 10` модель не знає більшості словоформ корпусу —
але це мала частка слововживань. Хвіст Ципфа довгий і легкий.

**02.** GloVe розкладає матрицю співвіднесень замість того, щоб ходити по парах.
Клітинок утричі менше, ніж пар, і в цьому вся економія. На нашому корпусі
різниця в якості між GloVe і Word2Vec менша за розкид між зернами.

**03.** Вагова функція GloVe — не косметика: без обрізання модель підлаштовується
під кілька службових пар, без ваги взагалі — під мільйон випадкових.

**04.** Субслова «впізнають» майже 90 % невідомих слів, але лема справді відома
лише в третини. Різниця майже втричі, і причина в тому, що спільними виявляються
закінчення, а не корені.

**05.** На самій задачі реконструкція з субслів дає невеликий, але вимірний
виграш — і водночас коштує в кілька разів більше часу та памʼяті, а на відомих
словах поступається звичайному Word2Vec.

**06.** Субслова зближують слова, схожі написанням, навіть коли за змістом вони
далекі. Це та сама властивість, що рятує морфологію, і в неї два боки.

**07.** І окремо — про сам замір. Вибір, яку з двох матриць вважати вектором
слова, змінив MRR сильніше, ніж вибір між GloVe і Word2Vec. Порівнюючи моделі,
спершу переконайся, що вони віддають однаковий бік розкладу, — інакше заміряєш
конвенцію, а не метод.

## Завдання

**🟢 Рівень 1.** Зміни `NGRAM_MIN` і `NGRAM_MAX` на 4-6 і перерахуй розділ 15:
як зміняться «оптимістична» оцінка й кількість субслів? Чи стане чесна оцінка
ближчою до оптимістичної?

**🟡 Рівень 2.** Побудуй той самий замір розділу 18, але замість леми
`pymorphy3` бери правильною відповіддю слово з тим самим першим складом.
Наскільки завищиться результат? Це вправа на те, як вибір «правильної відповіді»
керує висновком.

**🔴 Рівень 3.** Додай до `train_skipgram` **гешування субслів** у фіксовану
кількість комірок (як у справжньому FastText: `hash(ngram) % buckets`). Знайди
найменше число комірок, за якого MRR не падає більш ніж на розкид, і порахуй,
скільки памʼяті це заощаджує.

## 24 · Скільки все це коштувало

In [ ]:
print(f"увесь зошит: {time.process_time() - NOTEBOOK_START:.0f} с процесорного часу "
      f"на одному ядрі")
print("моделей навчено: 9 GloVe (три варіанти ваги × три зерна), "
      "3 Word2Vec, 3 FastText")